# Clipt Jersey OCR — PARSeq Fine-Tuning

Fine-tunes PARSeq scene-text recognition on football jersey crops to replace EasyOCR (~8% hit rate at 360p) with a domain-specific model targeting 70-85% per Koshkina et al. CVPR 2024 ([arXiv:2405.13896](https://arxiv.org/abs/2405.13896)).

**Pipeline:** [mkoshkina/jersey-number-pipeline](https://github.com/mkoshkina/jersey-number-pipeline) wraps PARSeq with a sports-specific data flow (LMDB packing, legibility classifier, hockey/SoccerNet pretrained weights).

**Inputs:** 720p clip URLs (output of `clipt_upscaling.ipynb`) + Roboflow API key for augmentation datasets.

**Output:** A `.ckpt` saved to Google Drive that drops into Railway at `app/model/parseq_clipt.ckpt` and replaces the EasyOCR call in `read_jersey_number()`.

**Expected wall time:** 30-60 min on A100 (data prep + 10 epochs on ~3,500 crops).

## Cell 1 — Verify GPU + clone pipeline

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — enable A100 in Runtime > Change runtime type'
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)

%cd /content
!git clone https://github.com/mkoshkina/jersey-number-pipeline.git jpipe || echo 'already cloned'
%cd /content/jpipe
!ls

## Cell 2 — Install dependencies

The Koshkina repo's `setup.py` clones PARSeq, ViTPose, Centroid-Reid, and downloads weights — so we let it do the heavy lifting. Then we add the supporting libs we need for crop generation + Cloudinary.

In [ ]:
%cd /content/jpipe
# pin to torch already on Colab; pipeline calls for pytorch 1.9 but newer works
!pip install -q timm pytorch-lightning lmdb hydra-core nltk
!pip install -q opencv-python pillow numpy tqdm
!pip install -q ultralytics roboflow easyocr cloudinary requests

# Run the pipeline's automated setup (clones PARSeq, downloads model weights from Drive).
# If this fails (Drive quota, network), we fall back to manual weight download below.
import subprocess
r = subprocess.run(['python3', 'setup.py'], capture_output=True, text=True, timeout=600)
print('setup stdout (last 500):', r.stdout[-500:])
print('setup stderr (last 500):', r.stderr[-500:])
print('rc:', r.returncode)

import os
os.makedirs('models', exist_ok=True)
print('\nmodels dir contents:')
!ls -la models/

# If hockey weights didn't land, fetch the originals from PARSeq repo as a fallback.
# These are the upstream pretrained weights on raw scene text — slightly worse starting
# point than hockey but always available.
if not any(f.endswith('.ckpt') for f in os.listdir('models')):
    print('\nFalling back to PARSeq upstream pretrained weights')
    !pip install -q parseq
    # PARSeq via torch.hub:
    import torch
    parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True)
    torch.save({'state_dict': parseq.state_dict()}, 'models/parseq_pretrained.ckpt')
    print('saved models/parseq_pretrained.ckpt')
    !ls -la models/

## Cell 3 — Mount Drive + configure

**Required:**
- `ROBOFLOW_API_KEY`: from <https://app.roboflow.com/settings/api>
- `CLIP_URLS`: Cloudinary URLs of upscaled clips (from `clipt_upscaling.ipynb`).
  Leave empty to train on Roboflow data only (still works, just less Clipt-specific).
- `TARGET_JERSEY`: Dustin = `'2'`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/data/crops', exist_ok=True)
os.makedirs('/content/data/crops_val', exist_ok=True)

# === CONFIGURE ===
ROBOFLOW_API_KEY = ''     # paste from app.roboflow.com/settings/api
TARGET_JERSEY    = '2'
EPOCHS           = 10
BATCH_SIZE       = 64

CLIP_URLS = [
    # paste from clipt_upscaling.ipynb output
    # 'https://res.cloudinary.com/dc33vjyyv/video/upload/v.../clip_01_1686s_720p.mp4',
]

assert ROBOFLOW_API_KEY, 'Set ROBOFLOW_API_KEY before continuing'
print(f'Target jersey: #{TARGET_JERSEY}')
print(f'Clipt clips supplied: {len(CLIP_URLS)}')
print(f'Training: {EPOCHS} epochs × batch {BATCH_SIZE}')

## Cell 4 — Pull Roboflow datasets

We pretrain on a base of ~3,500 labelled jersey crops from Roboflow Universe so PARSeq sees text variety beyond Dustin's #2. Cell 5 then fine-tunes on the actual Clipt clips on top.

In [ ]:
from roboflow import Roboflow
import os, glob

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rf_root = '/content/data/roboflow'
os.makedirs(rf_root, exist_ok=True)
os.chdir(rf_root)

# Workspace/project slugs may shift over time — if a download 404s,
# search Roboflow Universe for the named dataset and update the slug.
datasets = [
    ('football-tracking', 'football-jersey-tracker', 1),
    ('yakovk', 'jersey-numbers-i1wn5', 1),
]

for ws, proj, ver in datasets:
    try:
        d = rf.workspace(ws).project(proj).version(ver).download('folder')
        print(f'OK: {ws}/{proj} v{ver} → {d.location}')
    except Exception as e:
        print(f'SKIP: {ws}/{proj} v{ver} — {e}')

os.chdir('/content/jpipe')
print('\nRoboflow downloads:')
!find /content/data/roboflow -maxdepth 3 -type d

## Cell 5 — Generate jersey crops from Clipt clips

Uses YOLOv8 person detector to find players, then crops the torso region (where the jersey number is). Samples 6 frames per clip to capture different player poses.

**This is the manual-labelling boundary.** First-iteration crops are weakly labelled as `TARGET_JERSEY` because we know Dustin is in his clips — but other players in the frame will also be cropped. Open `/content/data/crops/` after this cell and reject obvious wrong-jersey crops before Cell 6.

In [ ]:
import cv2, requests, os
from pathlib import Path
from ultralytics import YOLO

yolo = YOLO('yolov8m.pt')

labels = []
crops_dir = Path('/content/data/crops')
crops_dir.mkdir(parents=True, exist_ok=True)

for i, url in enumerate(CLIP_URLS):
    clip_path = f'/content/clip_{i}.mp4'
    if not os.path.exists(clip_path):
        print(f'download {i}: {url[:60]}…')
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(clip_path, 'wb') as f:
                for chunk in r.iter_content(1024 * 1024):
                    f.write(chunk)

    cap = cv2.VideoCapture(clip_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        print(f'clip {i}: empty')
        continue
    sample_pcts = [0.1, 0.25, 0.4, 0.6, 0.75, 0.9]
    saved = 0
    for p in sample_pcts:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(total * p))
        ok, frame = cap.read()
        if not ok:
            continue
        # YOLO detect persons (class 0)
        results = yolo(frame, classes=[0], verbose=False)
        for j, box in enumerate(results[0].boxes.xyxy.cpu().numpy()):
            x1, y1, x2, y2 = map(int, box)
            h_box = y2 - y1
            # crop torso = upper 60% of person box (jersey number lives there)
            ty1 = y1 + int(h_box * 0.20)
            ty2 = y1 + int(h_box * 0.65)
            crop = frame[ty1:ty2, x1:x2]
            if crop.size == 0 or crop.shape[0] < 32 or crop.shape[1] < 24:
                continue
            fn = f'clip{i:02d}_p{int(p*100)}_box{j}.jpg'
            cv2.imwrite(str(crops_dir / fn), crop)
            labels.append((fn, TARGET_JERSEY))
            saved += 1
    cap.release()
    print(f'clip {i}: {saved} crops')

# write labels file
with open('/content/data/labels.txt', 'w') as f:
    f.write('\n'.join(f'{fn}\t{lbl}' for fn, lbl in labels))

n = len(labels)
print(f'\nTotal Clipt crops: {n}')
if n < 100 and CLIP_URLS:
    print('WARN: <100 Clipt crops. Re-run upscaling and supply more clips, or rely on Roboflow data alone.')
elif not CLIP_URLS:
    print('NOTE: training on Roboflow data only — generic jersey-number model, not Clipt-specific.')

## Cell 6 — Pack to LMDB + fine-tune PARSeq

PARSeq's training loop expects LMDB-packed data. The pipeline ships `tools/create_lmdb_dataset.py` (sourced from upstream PARSeq).

Training entrypoint: `main.py <DATASET> train --train_str` where `<DATASET>` is just an identifier in PARSeq's config registry. We register `Clipt` as a dataset by symlinking our LMDB into the expected location, then call train.

In [ ]:
%cd /content/jpipe
import os, glob, subprocess, shutil

# split labels 90/10 train/val
import random
random.seed(42)
with open('/content/data/labels.txt') as f:
    rows = [l for l in f.read().splitlines() if l.strip()]
random.shuffle(rows)
split = int(len(rows) * 0.9)
train_rows, val_rows = rows[:split], rows[split:]

for fn, lbl in [(r.split('\t')[0], r.split('\t')[1]) for r in val_rows]:
    src = f'/content/data/crops/{fn}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/data/crops_val/{fn}')
with open('/content/data/labels_train.txt', 'w') as f: f.write('\n'.join(train_rows))
with open('/content/data/labels_val.txt', 'w')   as f: f.write('\n'.join(val_rows))
print(f'train rows: {len(train_rows)}  val rows: {len(val_rows)}')

# Locate the create_lmdb_dataset helper. Path varies by repo layout.
candidates = glob.glob('/content/jpipe/**/create_lmdb_dataset.py', recursive=True)
print('lmdb helper candidates:', candidates)
lmdb_tool = candidates[0] if candidates else None

if lmdb_tool:
    !python {lmdb_tool} \
        --inputPath /content/data/crops \
        --gtFile /content/data/labels_train.txt \
        --outputPath /content/data/lmdb_train
    !python {lmdb_tool} \
        --inputPath /content/data/crops_val \
        --gtFile /content/data/labels_val.txt \
        --outputPath /content/data/lmdb_val
else:
    print('LMDB helper not found — clone PARSeq directly:')
    !git clone https://github.com/baudm/parseq /content/parseq || echo ok
    !python /content/parseq/tools/create_lmdb_dataset.py \
        --inputPath /content/data/crops \
        --gtFile /content/data/labels_train.txt \
        --outputPath /content/data/lmdb_train
    !python /content/parseq/tools/create_lmdb_dataset.py \
        --inputPath /content/data/crops_val \
        --gtFile /content/data/labels_val.txt \
        --outputPath /content/data/lmdb_val

print('LMDB packs:')
!ls -la /content/data/lmdb_train /content/data/lmdb_val

In [ ]:
# Fine-tune. Pretrained start: hockey weights if Cell 2 setup.py landed them,
# else generic PARSeq pretrained.
%cd /content/jpipe
import os, glob

ckpt = None
for cand in [
    'models/parseq_hockey.ckpt',
    'models/parseq_soccernet.ckpt',
    'models/parseq_pretrained.ckpt',
]:
    if os.path.exists(cand):
        ckpt = cand
        break
if ckpt is None:
    raise SystemExit('No starting checkpoint — re-run Cell 2')
print(f'Starting from: {ckpt}')

# Koshkina pipeline's training entrypoint. Note: PARSeq's hydra config registers
# datasets by name. The pipeline accepts SoccerNet/Hockey by default; for a
# custom dataset the cleanest path is to call PARSeq train directly with
# explicit lmdb paths.
#
# Try Koshkina wrapper first, fall back to direct PARSeq train.
import subprocess
wrapper_cmd = [
    'python3', 'main.py', 'Hockey', 'train', '--train_str',
    '--pretrained_str', ckpt,
    '--max_epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
]
r = subprocess.run(wrapper_cmd, capture_output=True, text=True, timeout=3600)
print('wrapper stdout (tail):', r.stdout[-800:])
print('wrapper stderr (tail):', r.stderr[-800:])
print('rc:', r.returncode)

if r.returncode != 0:
    print('\nWrapper failed — falling back to direct PARSeq training')
    !git clone https://github.com/baudm/parseq /content/parseq 2>/dev/null || true
    %cd /content/parseq
    !pip install -q -r requirements/core.txt 2>&1 | tail -5
    !python train.py +experiment=parseq-tiny \
        data.root_dir=/content/data \
        trainer.max_epochs={EPOCHS} \
        data.batch_size={BATCH_SIZE} \
        ckpt_path={ckpt}

## Cell 7 — Save trained checkpoint to Drive

PyTorch Lightning emits checkpoints under `lightning_logs/` or `outputs/`. We pick the most recent `.ckpt` and copy it to Drive for use on Railway.

In [ ]:
import glob, os, shutil, time

candidates = (
    glob.glob('/content/jpipe/**/last.ckpt', recursive=True) +
    glob.glob('/content/jpipe/**/*.ckpt', recursive=True) +
    glob.glob('/content/parseq/**/last.ckpt', recursive=True) +
    glob.glob('/content/parseq/**/*.ckpt', recursive=True)
)
candidates = [c for c in candidates if 'pretrained' not in c]

if not candidates:
    print('No fine-tuned checkpoint found — training likely failed. Check Cell 6 logs.')
else:
    candidates.sort(key=os.path.getmtime, reverse=True)
    latest = candidates[0]
    out = f'/content/drive/MyDrive/clipt_parseq_jersey_{TARGET_JERSEY}.ckpt'
    shutil.copy(latest, out)
    print(f'Latest: {latest}')
    print(f'Saved:  {out}  ({os.path.getsize(out)//1024//1024} MB)')

## Next steps — Wire into Railway

1. Download `clipt_parseq_jersey_2.ckpt` from Google Drive.
2. Copy to the Railway repo: `playerJerseyIdentification-master/app/model/parseq_clipt.ckpt`.
3. Replace EasyOCR call in `app/services/bytetrack_pipeline.py::read_jersey_number()`. PARSeq inference is heavier than EasyOCR but more accurate — accept the speed cost only after validating accuracy on a held-out clip.
4. Add the weight file to `Dockerfile` so the image bakes it in (same pattern as the YOLO weights already in `app/model/`).
5. Bump detection version to v8.33.0 and redeploy.

**Validation gate before merging:** run `/analyze-async` on a known clip and confirm jersey hit rate jumps from ~8% to ≥50%. If <50%, the dataset is too small or the labels are noisy — go back to Cell 5, label more carefully, and re-run.